# Clase 3 · Transformers y LLMs aplicados a reclamos y reseñas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hizocar/analitica-negocios-2/blob/main/docs/notebooks/clase3_llms.ipynb)

**Analítica para los Negocios II · USM** · Profesor: Sebastián Azócar

En este notebook pasas de la teoría de la cápsula a modelos de verdad, sobre tu caso (FinTrust o MercaSur):

| Parte | Qué haces | ¿Necesita clave de Gemini? |
|---|---|---|
| **0** | Cargar y limpiar los datos, partir en train/test | No |
| **A** | Contar **tokens reales** con tres tokenizadores (español vs. inglés) y estimar costos | No |
| **B** | Clasificar **zero-shot con un modelo abierto** de Hugging Face (mDeBERTa), sin API | No |
| **C** | Con **Gemini**: anonimizar, clasificar zero-shot y few-shot con salida estructurada, extraer campos en JSON, medir tokens, costo y latencia | **Sí** |
| **D** | Tabla comparativa, análisis de errores y ejercicios para el **Sprint 3** | No (usa lo que alcanzaste a correr) |

⏱️ **Tiempo estimado:** 60–90 minutos (buena parte es esperar a la API respetando los límites de la capa gratuita).

🔑 **Sin clave de Gemini el notebook no se cae:** las celdas de la parte C muestran cómo obtenerla y se saltan.
Puedes hacer las partes A, B y D igual.

> 💡 Ejecuta las celdas **en orden** (Entorno de ejecución → Ejecutar todas, o Shift+Enter una por una).

## 1. Instalación

Colab ya trae `transformers`, `torch`, `pandas`, `scikit-learn` y `pydantic`. Solo nos falta el SDK de Gemini
(`google-genai`).

In [ ]:
!pip -q install -U google-genai

## 2. Datos del caso y limpieza mínima

Elige tu caso en `CASO`. Todo el notebook funciona igual para ambos. La función `limpiar()` es la misma idea de
los Sprints 1 y 2: quita duplicados exactos y textos vacíos, y normaliza las categorías mal escritas
(«FRAUDE», «Credito y tasas», «Pagos»…).

⚠️ **Usa la misma partición train/test de los Sprints 1 y 2** (aquí: 20 % de test, estratificado, `random_state=42`).
Si en tus sprints usaste otra, cámbiala aquí para que la comparación sea justa.

In [ ]:
CASO = "fintrust"   # o "mercasur"

BASE = "https://raw.githubusercontent.com/hizocar/analitica-negocios-2/main/docs/datasets/"

import re, json, time, unicodedata, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 140)

CONFIG = {
    "fintrust": dict(
        archivo="reclamos_fintrust.csv", id="id_reclamo", etiqueta="categoria",
        clases=["Fraude", "Cobro indebido", "App y canales digitales",
                "Crédito y tasas", "Atención al cliente", "Información y trámites"],
        empresa="FinTrust, un banco chileno", unidad="reclamo", unidad_pl="reclamos",
        variantes={"cobros indebidos": "Cobro indebido"},
    ),
    "mercasur": dict(
        archivo="resenas_mercasur.csv", id="id_resena", etiqueta="aspecto",
        clases=["Despacho", "Calidad del producto", "Precio", "Atención",
                "Devoluciones y garantía", "Pago"],
        empresa="MercaSur, un retail chileno", unidad="reseña", unidad_pl="reseñas",
        variantes={"calidad producto": "Calidad del producto", "pagos": "Pago"},
    ),
}
cfg = CONFIG[CASO]
CLASES = cfg["clases"]

def clave(s):
    """'Crédito y Tasas ' -> 'credito y tasas' (minúsculas, sin tildes ni espacios extra)."""
    s = unicodedata.normalize("NFD", str(s).strip().lower())
    return "".join(ch for ch in s if unicodedata.category(ch) != "Mn")

CANON = {clave(c): c for c in CLASES} | cfg["variantes"]

def limpiar(df):
    df = df.drop_duplicates().copy()
    df["texto"] = df["texto"].fillna("").astype(str).str.strip()
    df = df[df["texto"].str.len() >= 10]                         # fuera "", "...", "reclamo", "ok"
    df["y"] = df[cfg["etiqueta"]].map(lambda c: CANON.get(clave(c)))
    return df.dropna(subset=["y"]).reset_index(drop=True)

crudo = pd.read_csv(BASE + cfg["archivo"])
df = limpiar(crudo)
train, test = train_test_split(df, test_size=0.2, stratify=df["y"], random_state=42)
print(f"{CASO}: {len(crudo)} filas crudas → {len(df)} limpias · train {len(train)} · test {len(test)}")
df["y"].value_counts()

## 3. Parte A · Tokens reales: ¿cuánto «pesa» el español?

En la cápsula usaste un tokenizador aproximado. Aquí usamos tres **reales**:

- **GPT-2** (2019): vocabulario aprendido casi solo con inglés.
- **GPT-4o** (2024): vocabulario moderno, mucho más multilingüe.
- **mDeBERTa**: el modelo abierto multilingüe que usarás en la parte B.

Comparamos frases de tu caso con su traducción al inglés.

In [ ]:
from transformers import AutoTokenizer

TOKENIZADORES = {
    "GPT-2 (inglés, 2019)": "gpt2",
    "GPT-4o (moderno)": "Xenova/gpt-4o",
    "mDeBERTa (multilingüe)": "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
}
toks = {}
for nombre, repo in TOKENIZADORES.items():
    try:
        toks[nombre] = AutoTokenizer.from_pretrained(repo)
    except Exception as e:
        print(f"⚠️ No se pudo cargar {nombre}: {e}")

PARES = [  # (español, inglés)
    ("Me cobraron dos veces la misma compra en Ripley.", "I was charged twice for the same purchase at Ripley."),
    ("Me clonaron la tarjeta y hay cargos que no reconozco.", "My card was cloned and there are charges I don't recognize."),
    ("La aplicación se cae cada vez que intento transferir.", "The app crashes every time I try to make a transfer."),
    ("El despacho llegó con cuatro días de atraso.", "The delivery arrived four days late."),
    ("Quiero devolver el producto porque vino fallado.", "I want to return the product because it arrived defective."),
]
filas = []
for es, en in PARES:
    fila = {"frase": es}
    for nombre, tk in toks.items():
        n_es, n_en = len(tk.tokenize(es)), len(tk.tokenize(en))
        fila[nombre] = f"{n_es} vs {n_en} ({n_es / n_en - 1:+.0%})"
    filas.append(fila)
print("Tokens: español vs inglés (y cuánto más gasta el español)")
pd.DataFrame(filas)

In [ ]:
# Así se ve un reclamo real partido en tokens (▁ o Ġ marcan el inicio de una palabra)
ejemplo = test["texto"].iloc[0]
print(ejemplo, "\n")
for nombre, tk in toks.items():
    piezas = tk.tokenize(ejemplo)
    print(f"{nombre:<24} {len(piezas):>3} tokens: {piezas[:18]}{' …' if len(piezas) > 18 else ''}")

**¿Cuánto costaría procesar todo el dataset?** Usamos el tokenizador de GPT-4o como aproximación (Gemini tiene el
suyo; con clave puedes contar exacto con `client.models.count_tokens`). Los precios son **supuestos editables**:
revisa los vigentes en <https://ai.google.dev/gemini-api/docs/pricing>.

In [ ]:
PRECIO_IN = 0.30    # US$ por millón de tokens de entrada (supuesto: gemini-2.5-flash, capa pagada)
PRECIO_OUT = 2.50   # US$ por millón de tokens de salida
INSTRUCCIONES = 300 # tokens fijos de un prompt zero-shot típico
SALIDA = 15         # tokens de una respuesta tipo {"categoria": "..."}

tk = toks.get("GPT-4o (moderno)") or next(iter(toks.values()))
n_tokens = df["texto"].map(lambda t: len(tk.tokenize(t)))
print(f"Tokens por {cfg['unidad']}: promedio {n_tokens.mean():.1f} · p95 {n_tokens.quantile(.95):.0f}")
costo = ((INSTRUCCIONES + n_tokens) * PRECIO_IN + SALIDA * PRECIO_OUT).sum() / 1e6
print(f"Clasificar las {len(df)} {cfg['unidad_pl']} una por una: ≈ US$ {costo:.2f}")
print(f"Proyección a 100.000 {cfg['unidad_pl']}/mes: ≈ US$ {costo / len(df) * 100_000:.2f}")

## 4. Parte B · Zero-shot con un modelo abierto (Hugging Face, sin API)

`MoritzLaurer/mDeBERTa-v3-base-mnli-xnli` es un **encoder** multilingüe (~280 millones de parámetros) entrenado para
*inferencia de lenguaje natural*: dado un texto (premisa) y una frase (hipótesis), estima si la hipótesis se
desprende del texto. El truco del *zero-shot*: para cada categoría armamos la hipótesis «Este reclamo trata sobre
{categoría}.» y elegimos la más probable. **No necesita datos etiquetados, ni clave, y los datos no salen de tu
máquina.**

⚠️ En CPU es lento (~0,5–1 s por texto con 6 categorías). Por eso usamos una muestra chica (`N_HF = 50`). Si activas
GPU en Colab (Entorno de ejecución → Cambiar tipo → T4 GPU) puedes subirla.

In [ ]:
import torch
from transformers import pipeline

N_HF = 50
muestra_hf = test.sample(N_HF, random_state=1)

zsc = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
               device=0 if torch.cuda.is_available() else -1)
plantilla = f"Este {cfg['unidad']} trata sobre {{}}." if CASO == "fintrust" else "Esta reseña trata sobre {}."

t0 = time.perf_counter()
salidas = zsc(muestra_hf["texto"].tolist(), candidate_labels=CLASES, hypothesis_template=plantilla, batch_size=8)
seg_hf = (time.perf_counter() - t0) / N_HF
pred_hf = [s["labels"][0] for s in salidas]

f1_hf = f1_score(muestra_hf["y"], pred_hf, average="macro", labels=CLASES)
print(f"mDeBERTa zero-shot sobre {N_HF} casos de test: accuracy {accuracy_score(muestra_hf['y'], pred_hf):.2f} "
      f"· F1 macro {f1_hf:.2f} · {seg_hf:.2f} s por caso")
pd.crosstab(muestra_hf["y"], pd.Series(pred_hf, index=muestra_hf.index, name="mDeBERTa"))

**Para pensar:** sin ver un solo ejemplo etiquetado, ¿qué tan lejos queda del TF-IDF del Sprint 1? ¿Qué categorías
confunde? Puedes probar etiquetas más descriptivas (por ejemplo, «cargos o compras que el cliente no reconoce» en
vez de «Fraude») o cambiar la plantilla de la hipótesis. Ojo: **no siempre mejora** (en nuestras pruebas con
FinTrust, las descripciones largas empeoraron el resultado). Por eso cada cambio se mide, no se supone.

## 5. Parte C · Gemini: configuración y clave

Para esta parte necesitas una **clave gratuita de la API de Gemini**:

1. Entra a <https://aistudio.google.com/apikey> con tu cuenta de Google y crea una clave.
2. En Colab, abre el panel **🔑 Secretos** (barra izquierda), agrega uno llamado `GEMINI_API_KEY` con tu clave y
   activa «Acceso al notebook».
3. Vuelve a ejecutar la celda de abajo.

Guía paso a paso: <https://hizocar.github.io/analitica-negocios-2/guias/gemini/>.
**Nunca pegues la clave dentro del código ni la subas al buzón.**

> 🔒 **Privacidad:** según los términos vigentes de la capa gratuita, Google puede usar el contenido que envías para
> mejorar sus productos (revisa <https://ai.google.dev/gemini-api/terms>). Razón de sobra para **anonimizar antes de
> enviar** (sección 6).

In [ ]:
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    import os; GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

MODELO = "gemini-2.5-flash"   # revisa el nombre vigente en https://ai.google.dev/gemini-api/docs/models
client = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None

if client is None:
    print("⏭️  No encontré GEMINI_API_KEY. Las celdas de Gemini se van a saltar (el notebook sigue funcionando).\n"
          "    Para activarlas: crea tu clave en https://aistudio.google.com/apikey, guárdala en 🔑 Secretos\n"
          "    con el nombre GEMINI_API_KEY y vuelve a ejecutar esta celda.")
else:
    print(f"✅ Cliente de Gemini listo · modelo {MODELO}")

### 5.1 Una función robusta para llamar a la API

La capa gratuita tiene **límites por minuto y por día** (pocas solicitudes por minuto). Si los superas, la API
responde con error **429**. Nuestra función:

- espera `PAUSA` segundos entre llamadas y **reintenta** con espera creciente si hay 429 o un error temporal (5xx);
- usa `temperature=0` (queremos consistencia) y **desactiva el «razonamiento»** (*thinking*) del modelo, que para
  clasificar solo suma latencia y tokens cobrados;
- pide **salida estructurada** con un esquema de Pydantic;
- mide **tokens** (`usage_metadata`) y **latencia**.

In [ ]:
from google.genai import errors

PAUSA = 7            # segundos entre llamadas (≈ 8 por minuto: bajo el límite gratuito típico)
MAX_REINTENTOS = 5
registro = []        # una fila por llamada: tokens, latencia, tarea

def llamar_gemini(prompt, esquema, tarea="", thinking=True):
    """Devuelve (objeto_parseado_o_None, texto_crudo). Registra tokens y latencia en `registro`."""
    if client is None:
        return None, None
    extra = dict(thinking_config=types.ThinkingConfig(thinking_budget=0)) if thinking else {}
    config = types.GenerateContentConfig(temperature=0, response_mime_type="application/json",
                                         response_schema=esquema, **extra)
    for intento in range(MAX_REINTENTOS):
        try:
            t0 = time.perf_counter()
            resp = client.models.generate_content(model=MODELO, contents=prompt, config=config)
            lat = time.perf_counter() - t0
            u = resp.usage_metadata
            registro.append(dict(tarea=tarea, latencia_s=lat,
                                 tokens_in=u.prompt_token_count or 0,
                                 tokens_out=(u.candidates_token_count or 0) + (u.thoughts_token_count or 0)))
            time.sleep(PAUSA)
            return resp.parsed, resp.text
        except errors.APIError as e:
            if "thinking" in str(e).lower() and thinking:      # modelo sin opción de desactivar thinking
                return llamar_gemini(prompt, esquema, tarea, thinking=False)
            if e.code == 429 or (e.code or 0) >= 500:
                espera = min(60, 10 * 2 ** intento)
                print(f"   ⏳ Error {e.code}: espero {espera} s y reintento ({intento + 1}/{MAX_REINTENTOS})")
                time.sleep(espera)
            else:
                print(f"   ❌ Error {e.code}: {e.message}")
                return None, None
    print("   ❌ Se agotaron los reintentos (¿llegaste al límite diario?).")
    return None, None

if client is None:
    print("⏭️  Sin clave: la función queda definida pero devolverá None.")

## 6. Anonimizar antes de enviar

Mismas reglas que en la cápsula: RUT, teléfonos, correos y nombres firmados con «Atte.». Enmascaramos **todos** los
textos que van a salir hacia la API (los de test y los ejemplos few-shot) y contamos cuántos tenían datos personales.

In [ ]:
PATRONES = {
    "RUT": r"\b\d{1,2}(?:\.+\d{3}){2}-[\dkK]\b|\b\d{7,8}-[\dkK]\b",
    "TELEFONO": r"(?:\+?56\s*)?\b9\s*\d{4}\s*\d{3,4}\b",
    "CORREO": r"[\w.+-]+@[\w-]+(?:\.+[\w-]+)+",
}
FIRMA = re.compile(r"(atte\.?\s+)([a-záéíóúñü]+(?:\s+[a-záéíóúñü]+)?)", flags=re.IGNORECASE)

def anonimizar(texto):
    encontrados = []
    for etiqueta, patron in PATRONES.items():
        texto, n = re.subn(patron, f"[{etiqueta}]", texto)
        encontrados += [etiqueta] * n
    texto, n = FIRMA.subn(r"\1[NOMBRE]", texto)
    return texto, encontrados + ["NOMBRE"] * n

for parte in (train, test):
    res = parte["texto"].map(anonimizar)
    parte["texto_anon"], parte["pii"] = res.str[0], res.str[1]

con_pii = test["pii"].str.len() > 0
print(f"Test: {con_pii.sum()} de {len(test)} textos con datos personales ({con_pii.mean():.1%})")
print("Por tipo:", pd.Series(test["pii"].sum()).value_counts().to_dict())
test.loc[con_pii, ["texto", "texto_anon"]].head(5)

## 7. Clasificación zero-shot con salida estructurada

Para no gastar el límite de solicitudes, enviamos los casos **en lotes de 10**: un solo prompt con 10 textos
(cada uno con su `id`) y pedimos una lista de `{id, categoria}`. Ventajas: 10 veces menos llamadas y las
instrucciones se pagan una vez por lote. Desventaja: si una llamada falla, pierdes 10 casos (que contamos como
inválidos).

El esquema usa `Literal` con tus categorías: el modelo **no puede** inventar otra.

In [ ]:
from typing import Literal, Optional
from pydantic import BaseModel, Field

Categoria = Literal[tuple(CLASES)]

class ItemClasificado(BaseModel):
    id: str
    categoria: Categoria

N_LLM = 100          # mínimo exigido en el Sprint 3: 100 casos de test
TAM_LOTE = 10
eval_llm = test.sample(N_LLM, random_state=7).copy()

INSTRUCCIONES_ZS = f"""Eres un analista de {cfg['unidad_pl']} de {cfg['empresa']}.
Clasifica cada {cfg['unidad']} en UNA de estas categorías: {", ".join(CLASES)}.
Si un texto menciona varios temas, elige el problema principal.
Devuelve una lista JSON con un objeto {{"id", "categoria"}} por cada texto, en el mismo orden."""

def prompt_lote(lote, instrucciones, ejemplos=""):
    casos = json.dumps([{"id": r.id_, "texto": r.texto_anon} for r in lote.itertuples()], ensure_ascii=False, indent=0)
    return f"{instrucciones}\n{ejemplos}\nTextos:\n{casos}"

eval_llm["id_"] = eval_llm[cfg["id"]].astype(str) + "_" + eval_llm.index.astype(str)   # id único
print(prompt_lote(eval_llm.head(2), INSTRUCCIONES_ZS))

In [ ]:
def clasificar(df_eval, instrucciones, ejemplos="", tarea="zero-shot"):
    """Clasifica por lotes. Devuelve una Serie con la categoría o None (inválida/faltante)."""
    pred = pd.Series(None, index=df_eval["id_"], dtype=object)
    for i in range(0, len(df_eval), TAM_LOTE):
        lote = df_eval.iloc[i:i + TAM_LOTE]
        parsed, crudo = llamar_gemini(prompt_lote(lote, instrucciones, ejemplos), list[ItemClasificado], tarea)
        if parsed:
            for item in parsed:
                if item.id in pred.index:
                    pred[item.id] = item.categoria
        print(f"   lote {i // TAM_LOTE + 1}/{-(-len(df_eval) // TAM_LOTE)} listo", end="\r")
    print()
    return pred

def evaluar(y_true, y_pred, nombre):
    y_pred = pd.Series(y_pred).fillna("INVÁLIDA").values
    return {"enfoque": nombre, "n": len(y_true),
            "accuracy": accuracy_score(y_true, y_pred),
            "F1 macro": f1_score(y_true, y_pred, average="macro", labels=CLASES),
            "% inválidas": 100 * np.mean(y_pred == "INVÁLIDA")}

resultados = []
if client is None:
    print("⏭️  Sin clave de Gemini: salto la clasificación zero-shot.")
else:
    eval_llm["pred_zs"] = clasificar(eval_llm, INSTRUCCIONES_ZS, tarea="zero-shot").values
    resultados.append(evaluar(eval_llm["y"], eval_llm["pred_zs"], "Gemini zero-shot"))
    print(pd.DataFrame(resultados).round(3).to_string(index=False))

## 8. Few-shot: con ejemplos del set de **entrenamiento**

Agregamos 2 ejemplos por categoría, tomados **solo de `train`** (nunca de test: sería hacer trampa) y anonimizados.
Elegimos ejemplos cortos y **claros**: si eliges al azar, puede tocarte un caso con la etiqueta mal puesta y le
enseñarías al modelo algo incorrecto. En tu proyecto puedes elegir ejemplos parecidos a los casos que el zero-shot
confundió.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Ejemplos "claros": casos que un modelo simple (TF-IDF) asigna con alta confianza a su propia etiqueta.
# Así evitamos usar como ejemplo un caso con etiqueta ruidosa (¡los datos traen ~6 %!).
cand = train[train["texto_anon"].str.len().between(40, 160)].drop_duplicates("texto_anon").copy()
vec_ej = TfidfVectorizer(ngram_range=(1, 2), min_df=2).fit(train["texto"])
lr_ej = LogisticRegression(max_iter=2000).fit(vec_ej.transform(train["texto"]), train["y"])
proba = lr_ej.predict_proba(vec_ej.transform(cand["texto"]))
cand["confianza"] = proba[np.arange(len(cand)), [list(lr_ej.classes_).index(y) for y in cand["y"]]]
ejemplos_df = cand[cand["confianza"] >= 0.8].groupby("y").sample(2, random_state=0)
EJEMPLOS = "Ejemplos resueltos:\n" + "\n".join(
    f'- "{r.texto_anon}" → {r.y}' for r in ejemplos_df.itertuples()) + "\n"
print(EJEMPLOS)

if client is None:
    print("⏭️  Sin clave de Gemini: salto la clasificación few-shot.")
else:
    eval_llm["pred_fs"] = clasificar(eval_llm, INSTRUCCIONES_ZS, EJEMPLOS, tarea="few-shot").values
    resultados.append(evaluar(eval_llm["y"], eval_llm["pred_fs"], "Gemini few-shot"))
    print(pd.DataFrame(resultados).round(3).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

col = "pred_fs" if "pred_fs" in eval_llm else ("pred_zs" if "pred_zs" in eval_llm else None)
if col is None:
    print("⏭️  Sin predicciones de Gemini: no hay matriz de confusión que mostrar.")
else:
    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay.from_predictions(eval_llm["y"], eval_llm[col].fillna("INVÁLIDA"),
                                            labels=CLASES + ["INVÁLIDA"], xticks_rotation=45, ax=ax,
                                            cmap="Blues", colorbar=False)
    ax.set_title(f"Gemini ({col.replace('pred_', '')}) · {N_LLM} casos de test")
    plt.tight_layout(); plt.show()

## 9. Extracción estructurada: varios campos en una sola llamada

Aquí el LLM brilla: en vez de solo una categoría, extraemos una **ficha** completa de cada caso. Esto sería muy
costoso de construir con modelos clásicos (necesitarías un modelo y datos etiquetados por campo).

Procesamos 20 casos, uno por llamada.

In [ ]:
class Ficha(BaseModel):
    categoria: Categoria
    urgencia: Literal["Alta", "Media", "Baja"]
    producto: str = Field(description="Producto o servicio mencionado; 'No indica' si no aparece")
    monto: Optional[int] = Field(None, description="Monto en pesos chilenos mencionado en el texto, sin puntos; null si no hay")
    requiere_contacto_urgente: bool = Field(description="True si hay riesgo de pérdida de dinero, fraude en curso o amenaza de escalar (SERNAC, cambiarse)")
    resumen_breve: str = Field(description="Resumen neutral de máximo 15 palabras")

PROMPT_FICHA = f"""Eres un analista de {cfg['unidad_pl']} de {cfg['empresa']}.
Lee el {cfg['unidad']} y completa la ficha. No inventes datos: si algo no aparece en el texto, usa null o 'No indica'.
{cfg['unidad'].capitalize()}:
<<<{{texto}}>>>"""

N_EXT = 20
fichas = []
if client is None:
    print("⏭️  Sin clave de Gemini: salto la extracción estructurada.")
else:
    for r in eval_llm.head(N_EXT).itertuples():
        f, _ = llamar_gemini(PROMPT_FICHA.format(texto=r.texto_anon), Ficha, tarea="extracción")
        if f:
            fichas.append({"texto": r.texto_anon, "y_real": r.y, **f.model_dump()})
    fichas = pd.DataFrame(fichas)
    # ¿El monto extraído aparece de verdad en el texto? (control contra alucinaciones)
    solo_digitos = lambda s: re.sub(r"\D", "", str(s))
    fichas["monto_en_texto"] = [pd.isna(m) or solo_digitos(int(m)) in solo_digitos(t)
                                for m, t in zip(fichas["monto"], fichas["texto"])]
    print(f"Categoría correcta: {(fichas['categoria'] == fichas['y_real']).mean():.0%} · "
          f"montos verificables en el texto: {fichas['monto_en_texto'].mean():.0%}")
    display(fichas)

## 10. Costo y latencia medidos

Con lo registrado en cada llamada (`usage_metadata` y el cronómetro) calculamos el **costo por 1.000 casos** y la
**latencia**. Ojo: en la capa gratuita no pagas, pero el costo estimado es el que tendrías en producción.

In [ ]:
if not registro:
    print("⏭️  No hay llamadas registradas (sin clave de Gemini).")
    costos = pd.DataFrame()
else:
    reg = pd.DataFrame(registro)
    casos_por_llamada = {"zero-shot": TAM_LOTE, "few-shot": TAM_LOTE, "extracción": 1}
    costos = reg.groupby("tarea").agg(llamadas=("latencia_s", "size"), tokens_in=("tokens_in", "mean"),
                                      tokens_out=("tokens_out", "mean"),
                                      lat_mediana_s=("latencia_s", "median"),
                                      lat_p95_s=("latencia_s", lambda s: s.quantile(.95)))
    costos["casos_por_llamada"] = costos.index.map(casos_por_llamada)
    costos["US$ por 1.000 casos"] = (1000 / costos["casos_por_llamada"]
                                    * (costos["tokens_in"] * PRECIO_IN + costos["tokens_out"] * PRECIO_OUT) / 1e6)
    costos["seg por caso"] = costos["lat_mediana_s"] / costos["casos_por_llamada"]
    display(costos.round(4))

## 11. Parte D · Tabla comparativa contra los Sprints 1 y 2

Para comparar en igualdad de condiciones, entrenamos aquí mismo el baseline TF-IDF + regresión logística (Sprint 1)
y lo evaluamos **sobre los mismos casos**. Completa la fila del Sprint 2 con tus resultados (embeddings +
clasificador).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
lr = LogisticRegression(max_iter=2000).fit(vec.fit_transform(train["texto"]), train["y"])
t0 = time.perf_counter()
pred_tfidf = lr.predict(vec.transform(eval_llm["texto"]))
seg_tfidf = (time.perf_counter() - t0) / len(eval_llm)

tabla = [evaluar(eval_llm["y"], pred_tfidf, "Sprint 1 · TF-IDF + logística") | {"US$ por 1.000": 0.0, "seg por caso": seg_tfidf, "etiquetas usadas": len(train)},
         # ✍️ Completa con tus números del Sprint 2 (sobre los mismos casos, idealmente)
         {"enfoque": "Sprint 2 · embeddings + clasificador", "n": N_LLM, "accuracy": np.nan, "F1 macro": np.nan,
          "% inválidas": 0.0, "US$ por 1.000": 0.0, "seg por caso": np.nan, "etiquetas usadas": len(train)},
         evaluar(muestra_hf["y"], pred_hf, "mDeBERTa zero-shot (50 casos)") | {"US$ por 1.000": 0.0, "seg por caso": seg_hf, "etiquetas usadas": 0}]

for r in resultados:
    tarea = "few-shot" if "few" in r["enfoque"] else "zero-shot"
    extra = {"etiquetas usadas": 0 if tarea == "zero-shot" else len(ejemplos_df)}
    if tarea in costos.index:
        extra |= {"US$ por 1.000": costos.loc[tarea, "US$ por 1.000 casos"], "seg por caso": costos.loc[tarea, "seg por caso"]}
    tabla.append(r | extra)

tabla = pd.DataFrame(tabla).set_index("enfoque")
tabla.round(3)

### 11.1 Análisis de errores

Mira los casos donde el LLM no coincide con la etiqueta. Para cada uno pregúntate: ¿es un **error del modelo**, un
**caso ambiguo** (menciona dos temas) o una **etiqueta ruidosa** (la etiqueta está mal y el LLM tiene razón)?
Recuerda que los datos traen ~6 % de etiquetas ruidosas.

In [ ]:
if col is None:
    print("⏭️  Sin predicciones de Gemini. Como alternativa, revisa los errores de mDeBERTa:")
    errores = muestra_hf.assign(pred=pred_hf).query("y != pred")[["texto", "y", "pred"]]
else:
    errores = eval_llm[eval_llm["y"] != eval_llm[col]][["texto_anon", "y"] +
                                                      [c for c in ["pred_zs", "pred_fs"] if c in eval_llm]]
    errores["pred_tfidf"] = pd.Series(pred_tfidf, index=eval_llm.index)[errores.index]
print(f"{len(errores)} errores")
errores.head(15)

## 12. ✍️ Ejercicios para tu proyecto (Sprint 3 · domingo 22 de noviembre, 23:59)

1. **Anonimización.** Revisa a mano 50 textos que `anonimizar()` dejó «limpios». ¿Encontraste datos personales que
   se escaparon? Mejora los patrones y reporta: % de textos con datos personales por tipo y qué riesgos quedan.
2. **Prompt zero-shot propio.** Agrega una definición de una línea por categoría (pensada para tu caso) y una regla
   de desempate para casos con dos temas. Compara el F1 macro contra el prompt base sobre los mismos 100 casos.
3. **Few-shot inteligente.** Elige los ejemplos entre los casos de `train` parecidos a los errores del zero-shot
   (pista: usa el buscador semántico del Sprint 2). ¿Mejora? ¿Cuánto cuesta en tokens?
4. **Extracción.** Ajusta la clase `Ficha` a tu caso (en MercaSur: aspecto, sentimiento, producto, precio,
   si requiere contacto urgente, resumen). Revisa 20 fichas a mano y reporta en qué campos acierta y en cuáles inventa.
5. **Tabla final.** Completa la tabla de la sección 11 con tu Sprint 2 y escribe 3–5 líneas, estilo piramidal:
   ¿cuándo conviene el LLM y cuándo el modelo clásico en tu caso? Considera calidad, costo, latencia, privacidad y
   datos etiquetados necesarios.
6. **Errores.** Comenta al menos 5 errores clasificándolos en error del modelo / caso ambiguo / etiqueta ruidosa.

---

## 🚀 Para ir más allá (opcional, no lo ejecutes por defecto): ajuste fino ligero

Si tienes miles de ejemplos etiquetados, otra opción es **ajustar** (*fine-tuning*) un encoder pequeño como
mDeBERTa a tus categorías. Suele igualar o superar al LLM en clasificación, corre en tu infraestructura y cuesta
casi nada por predicción. Requiere GPU (~10–20 min en una T4 de Colab). Un esqueleto con `transformers`:

```python
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)

base = "microsoft/mdeberta-v3-base"
tok = AutoTokenizer.from_pretrained(base)
id2label = dict(enumerate(CLASES)); label2id = {v: k for k, v in id2label.items()}
modelo = AutoModelForSequenceClassification.from_pretrained(base, num_labels=len(CLASES),
                                                            id2label=id2label, label2id=label2id)
def preparar(df):
    ds = Dataset.from_dict({"text": df["texto"].tolist(), "label": df["y"].map(label2id).tolist()})
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=128), batched=True)

args = TrainingArguments("ft-reclamos", num_train_epochs=2, per_device_train_batch_size=16,
                         learning_rate=3e-5, eval_strategy="epoch", report_to="none")
trainer = Trainer(model=modelo, args=args, train_dataset=preparar(train), eval_dataset=preparar(test),
                  processing_class=tok)
trainer.train()
```

Una alternativa con muy pocos ejemplos (8–16 por clase) es **SetFit**, que ajusta un modelo de
`sentence-transformers` como el de la Clase 2.